<a href="https://colab.research.google.com/github/Razeefshaik/MEvsAI/blob/main/notebooks/gpt_build.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("length of dataset in characters: ", len(text))

In [ ]:
print(text[:1000])

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

In [ ]:
%pip install sentencepiece

In [ ]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input="input.txt",
    model_prefix="tiny_shakespeare_bpe",
    model_type="bpe",
    vocab_size=1024,
    character_coverage=1.0,
    byte_fallback=True,
    normalization_rule_name="identity",
    add_dummy_prefix=False,
    remove_extra_whitespaces=False,
    bos_id=-1,
    eos_id=-1,
    pad_id=-1,
    hard_vocab_limit=False
)

sp = spm.SentencePieceProcessor(
    model_file="tiny_shakespeare_bpe.model"
)

vocab_size = sp.get_piece_size()

encode = lambda s: sp.encode(s, out_type=int)
decode = lambda ids: sp.decode(ids)

In [ ]:
text_sample = "I love machine learning"

print(sp.encode(text_sample, out_type=str))
print(encode(text_sample))
print(decode(encode(text_sample)))

In [ ]:
example = 'hii there!\nBefore we proceed.'
print('Text:   ', repr(example))
print('Pieces: ', sp.encode(example, out_type=str))
print('IDs:    ', encode(example))
print('Decoded:', repr(decode(encode(example))))
assert decode(encode(example)) == example

In [ ]:
import torch
data= torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

In [ ]:
n=int(0.9*len(data))
train_data= data[:n]
val_data= data[n:]

In [ ]:
block_size=128
train_data[: block_size+1]

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

In [ ]:
torch.manual_seed(1337)
batch_size= 4
block_size= 128

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


In [ ]:
print(m.parameters())

In [ ]:
optimizer= torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:

for steps in range(8000):
  xb, yb= get_batch('train')

  logits, loss= m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())

In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))

In [ ]:
m.eval()

prompt = "ROMEO:\n"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long
)

with torch.no_grad():
    generated = m.generate(
        idx=context,
        max_new_tokens=200
    )

print(decode(generated[0].tolist()))

In [ ]:
train_data = data[:n]
val_data = data[n:]

In [ ]:
@torch.no_grad()
def evaluate(split, num_batches=100):
    m.eval()
    losses = []

    for _ in range(num_batches):
        xb, yb = get_batch(split)
        logits, loss = m(xb, yb)
        losses.append(loss.item())

    return sum(losses) / len(losses)


train_loss = evaluate('train')
val_loss = evaluate('val')

print(f"Training Loss:   {train_loss:.4f}")
print(f"Validation Loss: {val_loss:.4f}")

In [ ]:
prompts = [
    "ROMEO:\n",
    "JULIET:\n",
    "KING:\n"
]

m.eval()

with torch.no_grad():
    for prompt in prompts:
        context = torch.tensor(
            [encode(prompt)],
            dtype=torch.long
        )

        output = m.generate(context, max_new_tokens=100)

        print("\nPROMPT:", repr(prompt))
        print(decode(output[0].tolist()))
        print("-" * 50)

In [ ]:
prompts = [
    "The king is here.",
    "My dog is here."
]

m.eval()

for prompt in prompts:
    ids = encode(prompt)
    context = torch.tensor([ids], dtype=torch.long)

    with torch.no_grad():
        logits, _ = m(context)
        probs = F.softmax(logits[:, -1, :], dim=-1)

    print(prompt)
    print("Last token ID:", ids[-1])
    print("Most likely next token:", decode([probs.argmax().item()]))

In [ ]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

In [ ]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape



```
 GPT
 ├── Token Embedding
 ├── Position Embedding
 ├── Block 1
 │    ├── MultiHeadAttention
 │    │    ├── Head 1
 │    │    ├── Head 2
 │    │    ├── Head 3
 │    │    └── Head 4
 │    ├── FeedFoward
 │    └── LayerNorm
 ├── Block 2
 ├── Block 3
 ├── Block 4
 ├── Final LayerNorm
 └── Language Modeling Head
```



In [ ]:
import math

@torch.no_grad()
def evaluate_model(model, num_batches=100, seed=42):
    was_training = model.training
    model.eval()

    results = {}

    for split in ['train', 'val']:
        data = train_data if split == 'train' else val_data

        # Separate random generator for reproducible evaluation
        rng = torch.Generator().manual_seed(seed)

        total_loss = 0.0
        total_tokens = 0

        for _ in range(num_batches):
            ix = torch.randint(
                len(data) - block_size,
                (batch_size,),
                generator=rng
            )

            x = torch.stack([
                data[i:i+block_size] for i in ix
            ]).to(device)

            y = torch.stack([
                data[i+1:i+block_size+1] for i in ix
            ]).to(device)

            logits, loss = model(x, y)

            token_count = y.numel()
            total_loss += loss.item() * token_count
            total_tokens += token_count

        avg_loss = total_loss / total_tokens

        results[split] = {
            "loss": avg_loss,
            "perplexity": math.exp(avg_loss)
        }

    model.train(was_training)
    return results

In [ ]:

import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
import matplotlib.pyplot as plt

# ============================================================
# 1. HYPERPARAMETERS
# ============================================================

batch_size = 16
block_size = 128
max_iters = 15000
eval_interval = 200
eval_iters = 100

learning_rate = 3e-4
min_learning_rate = 1e-5

n_embd = 128
n_head = 4
n_layer = 6
dropout = 0.1

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = "best_gpt_128d_6layer.pt"
tokenizer_path = "tiny_shakespeare_bpe.model"

torch.manual_seed(1337)

print("Device:", device)


# ============================================================
# 2. LOAD DATASET
# ============================================================

if not os.path.exists("input.txt"):
    import urllib.request

    url = (
        "https://raw.githubusercontent.com/karpathy/"
        "char-rnn/master/data/tinyshakespeare/input.txt"
    )
    urllib.request.urlretrieve(url, "input.txt")

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Dataset characters:", len(text))


# ============================================================
# 3. BPE TOKENIZATION
# ============================================================

# Reuse the existing tokenizer for fair comparison.
# Only train a new tokenizer if the file is missing.

if not os.path.exists(tokenizer_path):
    print("Training a new BPE tokenizer...")

    spm.SentencePieceTrainer.train(
        input="input.txt",
        model_prefix="tiny_shakespeare_bpe",
        model_type="bpe",
        vocab_size=1024,
        character_coverage=1.0,
        byte_fallback=True,
        normalization_rule_name="identity",
        add_dummy_prefix=False,
        remove_extra_whitespaces=False,
        bos_id=-1,
        eos_id=-1,
        pad_id=-1,
        hard_vocab_limit=False,
        minloglevel=2
    )

sp = spm.SentencePieceProcessor(
    model_file=tokenizer_path
)

vocab_size = sp.get_piece_size()

def encode(s):
    return sp.encode(s, out_type=int)

def decode(ids):
    return sp.decode(ids)

print("Vocabulary size:", vocab_size)

data = torch.tensor(
    encode(text),
    dtype=torch.long
)

# Same token-level split as the original notebook
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))


# ============================================================
# 4. BATCH GENERATION
# ============================================================

def get_batch(split):
    source = train_data if split == "train" else val_data

    ix = torch.randint(
        len(source) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        source[i:i + block_size] for i in ix
    ])

    y = torch.stack([
        source[i + 1:i + block_size + 1] for i in ix
    ])

    return x.to(device), y.to(device)


# ============================================================
# 5. REPRODUCIBLE EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_model(model, num_batches=eval_iters, seed=42):
    was_training = model.training
    model.eval()

    results = {}

    for split_idx, split in enumerate(["train", "val"]):
        source = train_data if split == "train" else val_data

        rng = torch.Generator().manual_seed(seed + split_idx)

        total_loss = 0.0
        total_tokens = 0

        for _ in range(num_batches):
            ix = torch.randint(
                len(source) - block_size,
                (batch_size,),
                generator=rng
            )

            x = torch.stack([
                source[i:i + block_size] for i in ix
            ]).to(device)

            y = torch.stack([
                source[i + 1:i + block_size + 1] for i in ix
            ]).to(device)

            _, loss = model(x, y)

            count = y.numel()
            total_loss += loss.item() * count
            total_tokens += count

        avg_loss = total_loss / total_tokens

        results[split] = {
            "loss": avg_loss,
            "perplexity": math.exp(avg_loss)
        }

    model.train(was_training)
    return results


# ============================================================
# 6. SINGLE SELF-ATTENTION HEAD
# ============================================================

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # Correct scaling: divide by sqrt(head_size)
        wei = (
            q @ k.transpose(-2, -1)
        ) * (k.shape[-1] ** -0.5)

        # Causal masking
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v

        return out


# ============================================================
# 7. MULTI-HEAD ATTENTION
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )

        return self.dropout(self.proj(out))


# ============================================================
# 8. FEED-FORWARD NETWORK
# ============================================================

class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# 9. TRANSFORMER BLOCK
# ============================================================

class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        assert n_embd % n_head == 0

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


# ============================================================
# 10. COMPLETE GPT MODEL
# ============================================================

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size, n_embd
        )

        self.position_embedding_table = nn.Embedding(
            block_size, n_embd
        )

        self.blocks = nn.Sequential(
            *[
                Block(n_embd, n_head)
                for _ in range(n_layer)
            ]
        )

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            flat_logits = logits.reshape(B * T, C)
            flat_targets = targets.reshape(B * T)

            loss = F.cross_entropy(
                flat_logits,
                flat_targets
            )

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx,
        max_new_tokens,
        temperature=0.8,
        top_k=40
    ):
        if temperature <= 0:
            raise ValueError("Temperature must be positive.")

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]

            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            # Temperature scaling
            logits = logits / temperature

            # Top-k filtering
            if top_k is not None:
                if top_k < 1:
                    raise ValueError("top_k must be >= 1.")

                k = min(top_k, logits.size(-1))

                values, _ = torch.topk(logits, k)

                cutoff = values[:, -1].unsqueeze(-1)

                logits = logits.masked_fill(
                    logits < cutoff,
                    float("-inf")
                )

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            idx = torch.cat((idx, idx_next), dim=1)

        return idx


# ============================================================
# 11. INITIALIZE A FRESH MODEL
# ============================================================

model = GPTLanguageModel().to(device)

num_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Model parameters: {num_params / 1e6:.3f} M")


# ============================================================
# 12. OPTIMIZER AND LEARNING-RATE SCHEDULER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max_iters,
    eta_min=min_learning_rate
)

best_val_loss = float("inf")
history = []


# ============================================================
# 13. TRAINING LOOP — FRESH START
# ============================================================

model.train()

for step in range(max_iters):

    # Sample batch
    xb, yb = get_batch("train")

    # Forward propagation
    logits, loss = model(xb, yb)

    # Reset gradients
    optimizer.zero_grad(set_to_none=True)

    # Backpropagation
    loss.backward()

    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    # Parameter update
    optimizer.step()

    # Learning-rate update
    scheduler.step()

    # Periodic evaluation
    if step % eval_interval == 0 or step == max_iters - 1:

        metrics = evaluate_model(model)

        train_loss = metrics["train"]["loss"]
        val_loss = metrics["val"]["loss"]

        train_ppl = metrics["train"]["perplexity"]
        val_ppl = metrics["val"]["perplexity"]

        current_lr = scheduler.get_last_lr()[0]

        history.append({
            "step": step,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_ppl": train_ppl,
            "val_ppl": val_ppl,
            "lr": current_lr
        })

        print(
            f"Step {step:5d} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f} | "
            f"Val PPL: {val_ppl:.2f} | "
            f"LR: {current_lr:.6f}"
        )

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "step": step,
                "val_loss": val_loss,
                "config": {
                    "vocab_size": vocab_size,
                    "batch_size": batch_size,
                    "block_size": block_size,
                    "n_embd": n_embd,
                    "n_head": n_head,
                    "n_layer": n_layer,
                    "dropout": dropout
                }
            }, checkpoint_path)

            print("  New best checkpoint saved!")


# ============================================================
# 14. PLOT TRAINING RESULTS
# ============================================================

steps = [h["step"] for h in history]

train_losses = [
    h["train_loss"] for h in history
]

val_losses = [
    h["val_loss"] for h in history
]

plt.figure(figsize=(10, 5))

plt.plot(
    steps,
    train_losses,
    label="Training Loss"
)

plt.plot(
    steps,
    val_losses,
    label="Validation Loss"
)

plt.xlabel("Training Steps")
plt.ylabel("Cross-Entropy Loss")
plt.title("GPT Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 15. LOAD THE BEST MODEL
# ============================================================

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Best checkpoint step:", checkpoint["step"])
print("Best validation loss:", checkpoint["val_loss"])


# ============================================================
# 16. FINAL EVALUATION
# ============================================================

final_metrics = evaluate_model(model)

for split, metrics in final_metrics.items():
    print(
        f"{split} | "
        f"Loss: {metrics['loss']:.4f} | "
        f"Perplexity: {metrics['perplexity']:.2f}"
    )


# ============================================================
# 17. GENERATE TEXT
# ============================================================

prompt = "ROMEO:\n"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

model.eval()

with torch.no_grad():
    generated = model.generate(
        context,
        max_new_tokens=500,
        temperature=0.8,
        top_k=40
    )

print("\n========== GENERATED TEXT ==========\n")
print(decode(generated[0].tolist()))


In [ ]:
model.load_state_dict(
    torch.load(
        "best_gpt.pt",
        map_location=device,
        weights_only=True
    )["model_state_dict"]
)

model.eval()

prompt = "ROMEO:\n"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

output = model.generate(
    context,
    max_new_tokens=300,
    temperature=0.5,
    top_k=40
)

print(decode(output[0].tolist()))

In [ ]:
import matplotlib.pyplot as plt

steps = [h['step'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]

plt.figure(figsize=(10, 5))

plt.plot(steps, train_losses, label='Training Loss')
plt.plot(steps, val_losses, label='Validation Loss')

plt.xlabel('Training Steps')
plt.ylabel('Cross-Entropy Loss')
plt.title('GPT Training Progress')

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
results = evaluate_model(model)

for split, metrics in results.items():
    print(
        f"{split}: "
        f"loss={metrics['loss']:.4f}, "
        f"perplexity={metrics['perplexity']:.2f}"
    )

In [ ]:
@torch.no_grad()
def generate_sample(model, prompt, max_new_tokens=100):
    model.eval()

    context = torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=device
    )

    output = model.generate(
        context,
        max_new_tokens=max_new_tokens
    )

    return decode(output[0].tolist())


print(generate_sample(model, "ROMEO:\n"))

In [ ]:
model.eval()

prompt = "ROMEO:\n"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

for temperature in [0.6, 0.8, 1.0]:

    torch.manual_seed(42)

    with torch.no_grad():
        output = model.generate(
            context,
            max_new_tokens=200,
            temperature=temperature,
            top_k=40
        )

    print(f"\n{'='*20}")
    print(f"TEMPERATURE = {temperature}")
    print(f"{'='*20}\n")

    print(decode(output[0].tolist()))